# Multi-GNN (PyG) on the IBM AML data – single notebook

Re-implementation of the **Multi-GIN** model from *Provably Powerful Graph Neural Networks for Directed Multigraphs* (Egressy et al., AAAI 2024) using **PyTorch Geometric**, wired to the `full_data.csv` produced by `data_processing.ipynb`.

It reuses the same PyG building blocks as the official `IBM/Multi-GNN` repo: `GINEConv`, `to_hetero(..., aggr='mean')` for reverse message passing, `LinkNeighborLoader` for sampling, and the `AddEgoIds` transform. The data prep (account/feature encoding, 60/20/20 temporal split, port numbers, z-normalisation) is re-written to start from your `full_data.csv` instead of the Kaggle `formatted_transactions.csv`, and does **not** need `datatable`.

| Adaptation | Flag | What it does |
|---|---|---|
| Reverse message passing | `reverse_mp` | heterogeneous graph with `to` and `rev_to` edges; one set of weights per direction, outputs averaged |
| Port numbering | `ports` | 2 extra edge features: the port the receiver gave the sender, and the port the sender gave the receiver |
| Ego IDs | `ego` | a 0/1 node feature marking the endpoints of the edges being classified |
| Edge updates (+EU) | `emlps` | MLP updates edge embeddings after every layer |

Task: edge classification (is this transaction laundering?). Metric: minority-class F1.

In [1]:
import time, itertools
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, precision_score, recall_score
from torch_geometric.data import Data, HeteroData
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GINEConv, BatchNorm, Linear, to_hetero
from torch_geometric.transforms import BaseTransform
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw): return it

# ---------------- paths (same variable as in data_processing.ipynb) ----------------
data_path = "/Users/trentbellinger/Desktop/CS 224W/Project/Data"   # REPLACE WITH YOUR LOCAL PATH
CSV = f"{data_path}/full_data.csv"

# ---------------- how much data ----------------
# Quick architecture test: only the first MAX_EDGES transactions (chronologically).
# None = whole file (~5M edges for HI-Small -> use a GPU).
MAX_EDGES = None

# ---------------- adaptations (Multi-GIN = reverse_mp + ports + ego) ----------------
CFG = dict(reverse_mp=True, ports=True, ego=True, emlps=False)

# ---------------- hyper-parameters ----------------
# GIN values from Multi-GNN/model_settings.json
HP = dict(lr=0.006213, n_hidden=66, n_gnn_layers=2, w_ce1=1.0, w_ce2=6.275, dropout=0.0098, final_dropout=0.105)
# The paper used batch_size=8192, num_neighbors=[100, 100] on a GPU. Smaller defaults so CPU runs finish.
TR = dict(batch_size=1024, num_neighbors=[10, 5], epochs=5)
SEED = 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import torch_geometric; print("device:", device, "| torch", torch.__version__, "| PyG", torch_geometric.__version__)

device: cpu | torch 2.10.0 | PyG 2.8.0.post1


## 1. Load and encode the data

In [2]:
df = pd.read_csv(CSV, dtype={"from_account": str, "to_account": str})
# parse timestamps explicitly: read_csv(parse_dates=...) silently leaves strings if the column mixes formats
try:
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed")
except (ValueError, TypeError):                      # pandas < 2.0 has no format="mixed" (and parses mixed formats itself)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)
if MAX_EDGES:
    df = df.iloc[:MAX_EDGES].copy()

def factorize_interleaved(a, b):
    # integer-encode two columns with one shared vocabulary, in order of first appearance
    # (row by row: a then b), like Multi-GNN's format_kaggle_files.py
    codes, _ = pd.factorize(np.column_stack([np.asarray(a, dtype=object), np.asarray(b, dtype=object)]).ravel())
    return codes[0::2].astype(np.int64), codes[1::2].astype(np.int64)

# an account = (bank, account number)
src, dst = factorize_interleaved(df.from_bank.astype(str) + "_" + df.from_account,
                                 df.to_bank.astype(str)   + "_" + df.to_account)
N_NODES = int(max(src.max(), dst.max())) + 1
recv_cur, _ = factorize_interleaved(df.receiving_currency, df.payment_currency)
fmt = pd.factorize(df.payment_format)[0]
t_sec = (df.timestamp - df.timestamp.min()).dt.total_seconds().to_numpy()

# the 4 edge features used by Multi-GNN: Timestamp, Amount Received, Received Currency, Payment Format
feat = np.stack([t_sec, df.amount_received.to_numpy(float), recv_cur, fmt], axis=1)
y = df.is_laundering.to_numpy().astype(np.int64)

print(f"{len(df):,} transactions | {N_NODES:,} accounts | illicit: {y.sum():,} ({y.mean()*100:.3f}%)")
print(f"time span: {t_sec.max()/86400:.1f} days")

/var/folders/j_/h9m1dhfn0g911dyhb57lr0vc0000gn/T/ipykernel_86847/814655234.py:1: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV, dtype={"from_account": str, "to_account": str})


5,078,345 transactions | 515,088 accounts | illicit: 5,177 (0.102%)
time span: 17.7 days


## 2. Temporal 60/20/20 split
Same search as the repo's `data_loading.py`: choose two cut points on day boundaries so that train/val/test proportions are closest to 60/20/20. The validation graph holds train+val edges and the test graph holds all edges, so patterns can use earlier transactions.

In [3]:
def temporal_split(t_sec, unit, props=(0.6, 0.2, 0.2)):
    unit_idx = (t_sec // unit).astype(int)
    counts = np.bincount(unit_idx, minlength=unit_idx.max() + 1)
    total, best, best_err = counts.sum(), None, np.inf
    cs = np.r_[0, np.cumsum(counts)]
    for i, j in itertools.combinations(range(len(counts)), 2):
        tot = np.array([cs[i], cs[j] - cs[i], total - cs[j]]) / total
        err = np.max(np.abs(tot - props) / props)
        if err < best_err: best, best_err = (i, j), err
    i, j = best
    return int(cs[i]), int(cs[j])          # number of edges in train / train+val

unit = 86400 if t_sec.max() >= 10 * 86400 else 3600      # days normally; hours for short quick tests
n_tr, n_trval = temporal_split(t_sec, unit)
n_all = len(df)
print(f"split unit: {'day' if unit == 86400 else 'hour'} | train {n_tr:,} | val {n_trval-n_tr:,} | test {n_all-n_trval:,}")
for name, sl in [("train", slice(0, n_tr)), ("val", slice(n_tr, n_trval)), ("test", slice(n_trval, n_all))]:
    print(f"  {name:5s} illicit ratio: {y[sl].mean()*100:.3f}%  ({y[sl].sum()} illicit)")

split unit: day | train 3,248,921 | val 965,524 | test 863,900
  train illicit ratio: 0.078%  (2530 illicit)
  val   illicit ratio: 0.107%  (1036 illicit)
  test  illicit ratio: 0.186%  (1611 illicit)


## 3. Port numbers
For an edge `u -> v`:
* **in-port** = the number node `v` gave to neighbour `u`; **out-port** = the number node `u` gave to neighbour `v`.
* Each node numbers its neighbours 0, 1, 2, … by the time of their first transaction with it, and parallel edges share a port.

This is a vectorised version of the repo's slow dict-based `ports()`. Ports are computed per graph (train / val / test) as in the repo.

In [4]:
def neighbor_ports(node, nbr, n_nodes):
    # edges must be in time order
    pair = node.astype(np.int64) * n_nodes + nbr
    uniq, first_idx, inv = np.unique(pair, return_index=True, return_inverse=True)
    inv = inv.reshape(-1)
    u_node = uniq // n_nodes
    order = np.lexsort((first_idx, u_node))                  # by node, then first appearance
    sorted_nodes = u_node[order]
    grp_start = np.r_[0, np.flatnonzero(np.diff(sorted_nodes)) + 1]
    grp_len = np.diff(np.r_[grp_start, len(order)])
    rank_sorted = np.arange(len(order)) - np.repeat(grp_start, grp_len)
    port_u = np.empty(len(uniq), dtype=np.int64); port_u[order] = rank_sorted
    return port_u[inv]

# sanity check on a hand-made multigraph (edges in time order)
_s = np.array([0, 1, 0, 2, 1, 0]); _d = np.array([3, 3, 3, 3, 0, 1])
assert neighbor_ports(_d, _s, 4).tolist() == [0, 1, 0, 2, 0, 0]   # in-ports; parallel edges 0->3 share port 0
assert neighbor_ports(_s, _d, 4).tolist() == [0, 0, 0, 0, 1, 1]   # out-ports
print("port numbering sanity check passed")

# (in-port, out-port) for the three graphs; computed once and reused by every configuration
def ports_for(n):
    return neighbor_ports(dst[:n], src[:n], N_NODES), neighbor_ports(src[:n], dst[:n], N_NODES)
t0 = time.time()
PORTS = {n: ports_for(n) for n in (n_tr, n_trval, n_all)}
print(f"ports computed in {time.time()-t0:.1f}s")

port numbering sanity check passed
ports computed in 5.5s


## 4. Build the PyG graph objects
* With `reverse_mp` a `HeteroData` with edge types `('node','to','node')` and `('node','rev_to','node')` (reversed edges). On the reverse edges the two port columns are swapped, so the *receiving* node's own port always comes first.
* Without it, a plain `Data`.
* Every edge carries its position in the graph as `eid`; the sampler slices it along with the other edge attributes, so a sampled batch can be mapped back to the original transactions.
* As in the repo, node features are a placeholder of ones, z-normalised (i.e. zeros), and edge features are z-normalised using each graph's own statistics.

In [5]:
def fwd(d):  return d["node", "to", "node"]     if isinstance(d, HeteroData) else d
def rev(d):  return d["node", "rev_to", "node"]
def node_store(d): return d["node"] if isinstance(d, HeteroData) else d

def make_graph(n, cfg):
    # graph made of the first n (time-ordered) edges
    in_p, out_p = PORTS[n]
    a = np.concatenate([feat[:n], in_p[:, None], out_p[:, None]], axis=1).astype(np.float64)
    sd = a.std(0); sd[sd == 0] = 1.0
    attr = torch.tensor((a - a.mean(0)) / sd, dtype=torch.float32)
    if not cfg["ports"]: attr = attr[:, :4]
    ei = torch.tensor(np.stack([src[:n], dst[:n]]))
    x = torch.zeros(N_NODES, 1)                         # z-normalised all-ones placeholder
    yy, eid = torch.tensor(y[:n]), torch.arange(n)
    if cfg["reverse_mp"]:
        d = HeteroData()
        d["node"].x = x
        f = d["node", "to", "node"]
        f.edge_index, f.edge_attr, f.y, f.eid = ei, attr, yy, eid
        r = d["node", "rev_to", "node"]
        r_attr = attr.clone()                           # clone: don't alias the forward edge features
        if cfg["ports"]: r_attr[:, [-2, -1]] = r_attr[:, [-1, -2]]
        r.edge_index, r.edge_attr, r.eid = ei.flip(0), r_attr, eid
        return d
    return Data(x=x, edge_index=ei, edge_attr=attr, y=yy, eid=eid)

def make_graphs(cfg):
    return make_graph(n_tr, cfg), make_graph(n_trval, cfg), make_graph(n_all, cfg)

G_tr, G_val, G_te = make_graphs(CFG)
print(G_tr)

HeteroData(
  node={ x=[515088, 1] },
  (node, to, node)={
    edge_index=[2, 3248921],
    edge_attr=[3248921, 6],
    y=[3248921],
    eid=[3248921],
  },
  (node, rev_to, node)={
    edge_index=[2, 3248921],
    edge_attr=[3248921, 6],
    eid=[3248921],
  }
)


### Sampler backend
`LinkNeighborLoader` needs either `pyg-lib` or `torch-sparse` installed (separately from PyG itself). The next cell checks for them. If neither is found it defines a drop-in replacement with the same batch format, built on numpy, so the notebook still runs. It samples up to `num_neighbors[k]` random incident edges per node at hop *k* (incoming edges, plus outgoing ones when `reverse_mp`), like PyG's sampler.

In [6]:
# a missing OR version-mismatched package can raise ImportError / OSError / RuntimeError, so catch them all
_HAS_SAMPLER = False
for _pkg in ("pyg_lib", "torch_sparse"):
    try:
        __import__(_pkg); _HAS_SAMPLER = True; break
    except Exception as _e:
        print(f"note: could not import {_pkg} ({type(_e).__name__}: {str(_e)[:90]})")

if _HAS_SAMPLER:
    print("using PyG's LinkNeighborLoader (pyg-lib / torch-sparse found)")
else:
    _v = ".".join(torch.__version__.split("+")[0].split(".")[:2]) + ".0"
    print("pyg-lib / torch-sparse not found -> using the built-in numpy neighbour sampler (same batch format).\n"
          f"To use PyG's own sampler instead:  pip install torch-sparse pyg-lib -f https://data.pyg.org/whl/torch-{_v}+cpu.html\n"
          "(use +cu118 / +cu121 etc. instead of +cpu on a CUDA machine; pyg-lib is optional if torch-sparse installs)")

    class LinkNeighborLoader:
        def __init__(self, data, num_neighbors, edge_label_index, edge_label, batch_size, shuffle=False, transform=None):
            self.data, self.num_neighbors, self.batch_size = data, num_neighbors, batch_size
            self.shuffle, self.transform = shuffle, transform
            self.hetero = isinstance(data, HeteroData)
            self.eli = edge_label_index[1] if self.hetero else edge_label_index
            self.lab = edge_label
            f = data["node", "to", "node"] if self.hetero else data
            self.src, self.dst = (t.numpy() for t in f.edge_index)
            n = (data["node"] if self.hetero else data).x.shape[0]
            self.csr = {}                                    # edges grouped by destination ('to') / source ('rev')
            for rel, key in (("to", self.dst), ("rev", self.src)):
                self.csr[rel] = (np.argsort(key, kind="stable"), np.r_[0, np.cumsum(np.bincount(key, minlength=n))])
            self.rng = np.random.default_rng()

        def _incident(self, rel, nodes, cap):
            order, ptr = self.csr[rel]
            starts = ptr[nodes]; cnt = ptr[nodes + 1] - starts; total = int(cnt.sum())
            if total == 0: return np.empty(0, dtype=np.int64)
            grp = np.repeat(np.arange(len(nodes)), cnt)
            offs = np.arange(total) - np.repeat(np.cumsum(cnt) - cnt, cnt)
            eids = order[starts[grp] + offs]
            if (cnt > cap).any():                            # keep `cap` random edges per node
                perm = np.lexsort((self.rng.random(total), grp)); eids = eids[perm[offs < cap]]
            return eids

        def __len__(self): return -(-self.eli.shape[1] // self.batch_size)

        def __iter__(self):
            n = self.eli.shape[1]
            idx = self.rng.permutation(n) if self.shuffle else np.arange(n)
            rels = ["to", "rev"] if self.hetero else ["to"]
            for i in range(0, n, self.batch_size):
                inp = idx[i:i + self.batch_size]; seed_ei = self.eli[:, inp].numpy()
                seed_nodes = np.unique(seed_ei); visited = frontier = seed_nodes; sel = {r: [] for r in rels}
                for cap in self.num_neighbors:
                    found = []
                    for r in rels:
                        e = self._incident(r, frontier, cap); sel[r].append(e)
                        found.append(self.src[e] if r == "to" else self.dst[e])
                    cand = np.unique(np.concatenate(found))
                    frontier = np.setdiff1d(cand, visited, assume_unique=True); visited = np.union1d(visited, frontier)
                ids = {r: np.unique(np.concatenate(sel[r])) for r in rels}
                nodes = np.unique(np.concatenate([seed_nodes] + [np.concatenate([self.src[e], self.dst[e]]) for e in ids.values()]))
                loc = lambda v: torch.as_tensor(np.searchsorted(nodes, v))
                xs = (self.data["node"] if self.hetero else self.data).x[torch.as_tensor(nodes)]
                inp_t = torch.as_tensor(inp)
                if self.hetero:
                    b = HeteroData(); b["node"].x = xs
                    for r, et in (("to", "to"), ("rev", "rev_to")):
                        if r not in ids: continue
                        e = torch.as_tensor(ids[r]); S = self.data["node", et, "node"]; T = b["node", et, "node"]
                        T.edge_index = torch.stack([loc(S.edge_index[0][e].numpy()), loc(S.edge_index[1][e].numpy())])
                        T.edge_attr, T.eid = S.edge_attr[e], S.eid[e]
                        if r == "to": T.y = S.y[e]
                    T = b["node", "to", "node"]
                    T.edge_label_index = torch.stack([loc(seed_ei[0]), loc(seed_ei[1])])
                    T.edge_label, T.input_id = self.lab[inp_t], inp_t
                else:
                    e = torch.as_tensor(ids["to"]); S = self.data
                    b = Data(x=xs, edge_index=torch.stack([loc(S.edge_index[0][e].numpy()), loc(S.edge_index[1][e].numpy())]),
                             edge_attr=S.edge_attr[e], y=S.y[e], eid=S.eid[e])
                    b.edge_label_index = torch.stack([loc(seed_ei[0]), loc(seed_ei[1])])
                    b.edge_label, b.input_id = self.lab[inp_t], inp_t
                yield self.transform(b) if self.transform is not None else b

using PyG's LinkNeighborLoader (pyg-lib / torch-sparse found)


## 5. Ego IDs, loaders and seed-edge handling
`LinkNeighborLoader` samples the neighbourhood around the endpoints of the *seed edges* of each batch. `AddEgoIds` (copied from the repo) appends a 0/1 ego feature to the endpoints of the seed edges.

One robustness addition compared with the repo: a seed edge is not guaranteed to be inside its own sampled neighbourhood. `attach_seeds` therefore removes any sampled copy and re-appends every seed edge (and its reverse) at the end of the batch, so the last `B` forward edges are always exactly the `B` seed edges.

In [7]:
class AddEgoIds(BaseTransform):
    # marks the endpoints of the seed edges (edge_label_index) with a 1 (from Multi-GNN/train_util.py)
    def forward(self, data):                      # newer PyG calls forward(), older versions call __call__()
        xs = node_store(data)
        ids = torch.zeros((xs.x.shape[0], 1), device=xs.x.device)
        ids[torch.unique(fwd(data).edge_label_index.view(-1))] = 1
        xs.x = torch.cat([xs.x, ids], dim=1)
        return data
    def __call__(self, data):
        return self.forward(data)

def make_loader(G, seed_ids, shuffle, cfg):
    ids = torch.as_tensor(seed_ids)
    f = fwd(G)
    ei, lab = f.edge_index[:, ids], f.y[ids]
    kw = dict(num_neighbors=TR["num_neighbors"], batch_size=TR["batch_size"], shuffle=shuffle,
              transform=AddEgoIds() if cfg["ego"] else None)
    if cfg["reverse_mp"]:
        return LinkNeighborLoader(G, edge_label_index=(("node", "to", "node"), ei), edge_label=lab, **kw)
    return LinkNeighborLoader(G, edge_label_index=ei, edge_label=lab, **kw)

def seed_ids_of(batch, seed_ids, cfg):
    inp = fwd(batch).input_id if cfg["reverse_mp"] else batch.input_id
    return torch.as_tensor(seed_ids)[inp.cpu()]

def attach_seeds(batch, G, gids, cfg):
    # put the seed edges (global ids `gids`) last in the batch; returns their labels
    fb, fg = fwd(batch), fwd(G)
    seed_ei = fb.edge_label_index
    keep = ~torch.isin(fb.eid, gids)
    fb.edge_index = torch.cat([fb.edge_index[:, keep], seed_ei], dim=1)
    fb.edge_attr  = torch.cat([fb.edge_attr[keep], fg.edge_attr[gids]], dim=0)
    if cfg["reverse_mp"]:
        rb, rg = rev(batch), rev(G)
        keep = ~torch.isin(rb.eid, gids)
        rb.edge_index = torch.cat([rb.edge_index[:, keep], seed_ei.flip(0)], dim=1)
        rb.edge_attr  = torch.cat([rb.edge_attr[keep], rg.edge_attr[gids]], dim=0)
    return fg.y[gids]

def model_inputs(batch, cfg):
    if cfg["reverse_mp"]:
        return batch.x_dict, batch.edge_index_dict, batch.edge_attr_dict
    return batch.x, batch.edge_index, batch.edge_attr

## 6. The model
`GINe` is copied from `Multi-GNN/models.py` (GIN with edge features, residual node updates, optional edge-update MLPs, readout `[h_src, h_dst, h_edge]` → MLP). With `reverse_mp`, `to_hetero(model, metadata, aggr='mean')` duplicates the edge-level modules per direction and averages the node updates – exactly as in the official code.

In [8]:
class GINe(torch.nn.Module):
    def __init__(self, num_features, num_gnn_layers, n_classes=2, n_hidden=100, edge_updates=False,
                 residual=True, edge_dim=None, dropout=0.0, final_dropout=0.5):
        super().__init__()
        self.n_hidden, self.num_gnn_layers = n_hidden, num_gnn_layers
        self.edge_updates, self.final_dropout = edge_updates, final_dropout
        self.node_emb = nn.Linear(num_features, n_hidden)
        self.edge_emb = nn.Linear(edge_dim, n_hidden)
        self.convs, self.emlps, self.batch_norms = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        for _ in range(self.num_gnn_layers):
            conv = GINEConv(nn.Sequential(nn.Linear(n_hidden, n_hidden), nn.ReLU(), nn.Linear(n_hidden, n_hidden)),
                            edge_dim=n_hidden)
            if self.edge_updates:
                self.emlps.append(nn.Sequential(nn.Linear(3 * n_hidden, n_hidden), nn.ReLU(), nn.Linear(n_hidden, n_hidden)))
            self.convs.append(conv)
            self.batch_norms.append(BatchNorm(n_hidden))
        self.mlp = nn.Sequential(Linear(n_hidden * 3, 50), nn.ReLU(), nn.Dropout(self.final_dropout),
                                 Linear(50, 25), nn.ReLU(), nn.Dropout(self.final_dropout), Linear(25, n_classes))

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)
        for i in range(self.num_gnn_layers):
            x = (x + F.relu(self.batch_norms[i](self.convs[i](x, edge_index, edge_attr)))) / 2
            if self.edge_updates:
                edge_attr = edge_attr + self.emlps[i](torch.cat([x[src], x[dst], edge_attr], dim=-1)) / 2
        x = x[edge_index.T].reshape(-1, 2 * self.n_hidden).relu()
        x = torch.cat((x, edge_attr.view(-1, edge_attr.shape[1])), 1)
        return self.mlp(x)

def build_model(cfg, hp, metadata):
    model = GINe(num_features=2 if cfg["ego"] else 1, num_gnn_layers=hp["n_gnn_layers"], n_classes=2,
                 n_hidden=round(hp["n_hidden"]), residual=False, edge_updates=cfg["emlps"],
                 edge_dim=6 if cfg["ports"] else 4, dropout=hp["dropout"], final_dropout=hp["final_dropout"])
    if cfg["reverse_mp"]:
        model = to_hetero(model, metadata, aggr="mean")
    return model

m = build_model(CFG, HP, G_tr.metadata() if CFG["reverse_mp"] else None)
print("parameters:", sum(p.numel() for p in m.parameters()))

parameters: 77004


## 7. Train / evaluate
Smoke test first: one batch through the loader, to check the seed handling before spending time on training.

In [9]:
loader = make_loader(G_tr, np.arange(n_tr), True, CFG)
b = next(iter(loader))
gids = seed_ids_of(b, np.arange(n_tr), CFG)
labels = attach_seeds(b, G_tr, gids, CFG)
B = len(gids)
print(b)
assert fwd(b).edge_index.shape[1] == fwd(b).edge_attr.shape[0]
assert torch.equal(fwd(b).edge_attr[-B:], fwd(G_tr).edge_attr[gids]), "seed edges are not the last B edges"
x_, ei_, ea_ = model_inputs(b, CFG)
m = build_model(CFG, HP, G_tr.metadata() if CFG["reverse_mp"] else None)
out = m(x_, ei_, ea_)
out = out[("node", "to", "node")] if CFG["reverse_mp"] else out
print("seeds:", B, "| logits for all forward edges:", tuple(out.shape), "| seed labels:", labels.sum().item(), "illicit")

HeteroData(
  node={
    x=[17442, 2],
    n_id=[17442],
    num_sampled_nodes=[3],
  },
  (node, to, node)={
    edge_index=[2, 34874],
    edge_attr=[34874, 6],
    y=[34518],
    eid=[34518],
    e_id=[34518],
    num_sampled_edges=[2],
    input_id=[1024],
    edge_label_index=[2, 1024],
    edge_label=[1024],
  },
  (node, rev_to, node)={
    edge_index=[2, 30750],
    edge_attr=[30750, 6],
    eid=[30262],
    e_id=[30262],
    num_sampled_edges=[2],
  }
)
seeds: 1024 | logits for all forward edges: (34874, 2) | seed labels: 0 illicit


In [10]:
def scores(y_true, y_pred):
    return dict(f1=f1_score(y_true, y_pred, zero_division=0), prec=precision_score(y_true, y_pred, zero_division=0),
                rec=recall_score(y_true, y_pred, zero_division=0))

def forward_seeds(model, batch, G, seed_ids, cfg):
    gids = seed_ids_of(batch, seed_ids, cfg)
    labels = attach_seeds(batch, G, gids, cfg)
    batch = batch.to(device)
    out = model(*model_inputs(batch, cfg))
    out = out[("node", "to", "node")] if cfg["reverse_mp"] else out
    return out[-len(gids):], labels.to(device), gids          # seed edges are the last B

@torch.no_grad()
def evaluate(model, loader, G, seed_ids, cfg):
    model.eval(); preds, gids_all = [], []
    for batch in loader:
        out, _, gids = forward_seeds(model, batch, G, seed_ids, cfg)
        preds.append(out.argmax(-1).cpu()); gids_all.append(gids)
    return torch.cat(preds).numpy(), torch.cat(gids_all).numpy()

def run(cfg=CFG, hp=HP, tr=TR, seed=SEED, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed)
    G_tr, G_val, G_te = make_graphs(cfg)
    tr_ids, val_ids, te_ids = np.arange(n_tr), np.arange(n_tr, n_trval), np.arange(n_trval, n_all)
    tr_loader  = make_loader(G_tr,  tr_ids,  True,  cfg)
    val_loader = make_loader(G_val, val_ids, False, cfg)
    te_loader  = make_loader(G_te,  te_ids,  False, cfg)
    model = build_model(cfg, hp, G_tr.metadata() if cfg["reverse_mp"] else None).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=hp["lr"])
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([hp["w_ce1"], hp["w_ce2"]], dtype=torch.float32, device=device))
    best = dict(val_f1=-1)
    for epoch in range(tr["epochs"]):
        model.train(); t0 = time.time(); losses, tp, tt = [], [], []
        for batch in tqdm(tr_loader, desc=f"epoch {epoch+1}", leave=False, disable=not verbose):
            out, labels, _ = forward_seeds(model, batch, G_tr, tr_ids, cfg)
            loss = loss_fn(out, labels)
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item()); tp.append(out.argmax(-1).cpu()); tt.append(labels.cpu())
        tr_f1 = f1_score(torch.cat(tt).numpy(), torch.cat(tp).numpy(), zero_division=0)
        val_pred, _ = evaluate(model, val_loader, G_val, val_ids, cfg)
        te_pred, te_gids = evaluate(model, te_loader, G_te, te_ids, cfg)
        v, t = scores(y[val_ids], val_pred), scores(y[te_gids], te_pred)
        if v["f1"] > best["val_f1"]:
            best = dict(epoch=epoch + 1, val_f1=v["f1"], test=t, test_pred=te_pred, test_ids=te_gids)
        if verbose:
            print(f"epoch {epoch+1:3d} | loss {np.mean(losses):.4f} | train F1 {tr_f1:.3f} | val F1 {v['f1']:.3f} "
                  f"| test F1 {t['f1']:.3f} (P {t['prec']:.3f} R {t['rec']:.3f}) | {time.time()-t0:.0f}s")
    print(f"best val F1 {best['val_f1']:.4f} at epoch {best['epoch']} -> test F1 {best['test']['f1']:.4f} "
          f"(P {best['test']['prec']:.3f}, R {best['test']['rec']:.3f})")
    return model, best

In [11]:
model, best = run(CFG)

epoch 1:   0%|          | 0/3173 [00:00<?, ?it/s]

epoch   1 | loss 0.0212 | train F1 0.020 | val F1 0.223 | test F1 0.301 (P 0.420 R 0.235) | 414s


epoch 2:   0%|          | 0/3173 [00:00<?, ?it/s]

epoch   2 | loss 0.0173 | train F1 0.178 | val F1 0.259 | test F1 0.409 (P 0.412 R 0.407) | 424s


epoch 3:   0%|          | 0/3173 [00:00<?, ?it/s]

epoch   3 | loss 0.0169 | train F1 0.209 | val F1 0.287 | test F1 0.359 (P 0.440 R 0.304) | 432s


epoch 4:   0%|          | 0/3173 [00:00<?, ?it/s]

epoch   4 | loss 0.0157 | train F1 0.239 | val F1 0.294 | test F1 0.414 (P 0.647 R 0.305) | 439s


epoch 5:   0%|          | 0/3173 [00:00<?, ?it/s]

epoch   5 | loss 0.0163 | train F1 0.244 | val F1 0.228 | test F1 0.255 (P 0.824 R 0.151) | 443s
best val F1 0.2944 at epoch 4 -> test F1 0.4143 (P 0.647, R 0.305)


## 8. Recall per laundering pattern (uses the `laundering_type` column from your notebook)

In [12]:
res = df.iloc[best["test_ids"]].copy()          # test_ids are positions in the (time-sorted) dataframe
res["pred"] = best["test_pred"]
ill = res[res.is_laundering == 1].copy()
ill["pattern"] = ill.laundering_type.fillna("(illicit, not in a pattern)")
print(ill.groupby("pattern").agg(n=("pred", "size"), recall=("pred", "mean")).round(3))

                               n  recall
pattern                                 
(illicit, not in a pattern)  371   0.000
Bipartite                     66   0.136
Cycle                         99   0.141
Fan-In                       123   0.252
Fan-Out                      131   0.687
Gather-Scatter               381   0.580
Random                        84   0.155
Scatter-Gather               239   0.435
Stack                        117   0.077


## 9. Optional: ablation (the paper's cumulative-adaptation rows)
Set `RUN_ABLATION = True`. Each row is a full training run, so it uses fewer epochs.

In [ ]:
RUN_ABLATION = False
if RUN_ABLATION:
    variants = {
        "GIN":                 dict(reverse_mp=False, ports=False, ego=False, emlps=False),
        "+ReverseMP":          dict(reverse_mp=True,  ports=False, ego=False, emlps=False),
        "+Ports":              dict(reverse_mp=True,  ports=True,  ego=False, emlps=False),
        "+EgoIDs (Multi-GIN)": dict(reverse_mp=True,  ports=True,  ego=True,  emlps=False),
        "Multi-GIN+EU":        dict(reverse_mp=True,  ports=True,  ego=True,  emlps=True),
    }
    rows = []
    for name, c in variants.items():
        print(f"\n=== {name} ===")
        _, b = run(c, tr=dict(TR, epochs=3), verbose=False)
        rows.append(dict(model=name, val_f1=b["val_f1"], test_f1=b["test"]["f1"], test_prec=b["test"]["prec"], test_rec=b["test"]["rec"]))
    print(pd.DataFrame(rows).round(3).to_string(index=False))